# ECG Experiment Notebook
Generated on 2026-08-06T21:41:04.231084
Balance mode: **binary**

In [ ]:

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ecg_pipeline.dataset_factory import DatasetFactory
from ecg_pipeline.temporal_encoder.predictor import TemporalPredictor
import mlflow.pytorch


In [ ]:

# Load the PTB‑XL dataset with the selected balance mode
_, val_ds, test_ds, loader = DatasetFactory.create_datasets(
    dataset_type="ptbxl",
    download=False,
    resolution="lr",
    balance_mode="binary"
)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4)


In [ ]:

def load_best_model(experiment_name: str, balance_mode: str):
    exp = mlflow.get_experiment_by_name(experiment_name)
    if exp is None:
        raise ValueError(f"Experiment {experiment_name} not found")
    runs = mlflow.search_runs([exp.experiment_id])
    runs = runs[runs["params.balance_mode"] == balance_mode]
    runs = runs[runs["status"] == "FINISHED"]
    if runs.empty:
        raise ValueError(f"No finished runs for {experiment_name}:{balance_mode}")
    run_id = runs.sort_values("start_time", ascending=False).iloc[0]["run_id"]
    model = mlflow.pytorch.load_model(f"runs:/{run_id}/model")
    model.eval()
    return model


In [ ]:

resnet = load_best_model("ECG_ResNet_Final", "{balance_mode}")
transformer = load_best_model("ECG_Transformer_Final", "{balance_mode}")


In [ ]:

resnet_pred = TemporalPredictor(resnet, device='cuda' if torch.cuda.is_available() else 'cpu')
transformer_pred = TemporalPredictor(transformer, device='cuda' if torch.cuda.is_available() else 'cpu')
resnet_probs = resnet_pred.predict_proba(test_loader)
transformer_probs = transformer_pred.predict_proba(test_loader)
# Simple average ensemble (can be adjusted later)
ensemble_probs = 0.5 * resnet_probs + 0.5 * transformer_probs
preds = (ensemble_probs >= 0.5).astype(int)


In [ ]:

# Placeholder: replace with actual per‑class metric calculation
print("Ensemble predictions shape:", preds.shape)
